In [10]:
import logging
from pathlib import Path

import pandas as pd
from _pandas_patcher import PrintAsImageString
from rdkit import Chem
from stereomolgraph import StereoMolGraph
from stereomolgraph.algorithms.symmetry import topological_symmetry_number


def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the SI project root.")


logging.getLogger("_pandas_patcher").setLevel(logging.ERROR)

In [11]:
project_root = find_project_root()
csv_path = project_root / "data" / "kinbot.csv"

df_input = pd.read_csv(csv_path, escapechar="\\").assign(
    kinbot_sym_num=lambda df: (
        df["kinbot_value"].combine_first(df["expected_value"]).astype("Int64")
    ),
)

rows = []
for inchi, kinbot_sym_num in df_input[["inchi", "kinbot_sym_num"]].itertuples(
    index=False
):
    rdmol = Chem.MolFromInchi(
        inchi, sanitize=True, removeHs=False, treatWarningAsError=True
    )
    if rdmol is not None:
        rdmol = Chem.AddHs(rdmol)

    if rdmol is None:
        rows.append(
            {
                "stereomolgraph": None,
                "inchi": inchi,
                "kinbot_sym_num": kinbot_sym_num,
                "top_sym_num": None,
            }
        )
        continue

    graph = StereoMolGraph.from_rdmol(
        rdmol, stereo_complete=True, resonance=True, lone_pair_stereo=True
    )

    rows.append(
        {
            "stereomolgraph": graph,
            "inchi": inchi,
            "kinbot_sym_num": kinbot_sym_num,
            "top_sym_num": topological_symmetry_number(graph),
        }
    )

df_symmetry = pd.DataFrame(rows)

# --- Merge sigma symmetry numbers from sigma.csv ---
sigma_path = project_root / "data" / "sigma.csv"
df_sigma = pd.read_csv(sigma_path)
df_sigma["inchi"] = df_sigma["inchi"].str.strip()
df_symmetry["inchi"] = df_symmetry["inchi"].str.strip()
df_symmetry = df_symmetry.merge(
    df_sigma[["inchi", "sigma_calculated"]],
    on="inchi",
    how="left",
).rename(columns={"sigma_calculated": "sigma_sym_num"})

df_symmetry["sigma_sym_num"] = df_symmetry["sigma_sym_num"].astype("Int64")
# ---

mismatch = df_symmetry["kinbot_sym_num"].ne(df_symmetry["top_sym_num"])
both_missing = df_symmetry["kinbot_sym_num"].isna() & df_symmetry["top_sym_num"].isna()
df_symmetry = (
    df_symmetry.assign(_mismatch=mismatch & ~both_missing)
    .sort_values(["_mismatch", "inchi"], ascending=[False, True])
    .drop(columns="_mismatch")
    .reset_index(drop=True)
)

In [12]:
from IPython.display import HTML, display

styled = (
    df_symmetry[
        [
            "stereomolgraph",
            "inchi",
            "top_sym_num",
            "kinbot_sym_num",
            "sigma_sym_num",
        ]
    ]
    .style.hide(axis="index")
    .format(
        {
            "stereomolgraph": PrintAsImageString,
        },
        escape=None,
    )
    .set_properties(**{"text-align": "left", "vertical-align": "middle"})
    .set_table_styles(
        [
            {"selector": "table", "props": [("border-collapse", "collapse")]},
            {
                "selector": "th:not(:last-child), td:not(:last-child)",
                "props": [("border-right", "1px solid #b0b0b0")],
            },
            {"selector": "th", "props": [("text-align", "left")]},
        ]
    )
)

table_html = styled.to_html()
html_document = (
    "<!DOCTYPE html><html lang='en'><head><meta charset='utf-8'>"
    "<title>Kinbot &amp; Sigma Symmetry Numbers</title></head><body>"
    f"<div style='overflow-x:auto'>{table_html}</div></body></html>"
)
output_html_path = find_project_root() / "reports" / "kinbot.html"
output_html_path.parent.mkdir(parents=True, exist_ok=True)
output_html_path.write_text(html_document, encoding="utf-8")

display(HTML(f"<div style='overflow-x:auto'>{table_html}</div>"))
print(f"Saved HTML report to: {output_html_path.resolve()}")

Saved HTML report to: C:\Users\maxim\Code\experiments\SI_SymmetryNumbers\reports\kinbot.html
